In [231]:
import pandas as pd
import numpy as np
import re,yaml,os
import itertools as it
import networkx as nx
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [232]:
pd.__version__
%store -r numOMP
num=numOMP
# num=0
print(num)

2


In [233]:
rep="/Volumes/BroadExt/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
rep="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
fParadigmes="vlexique2-CV3-Test%d.csv"%num
# fParadigmes="vlexique2-S%d.csv"%num
paradigmes=pd.read_csv(rep+fParadigmes,sep=";",encoding="utf8")

In [234]:
cols=paradigmes.columns.tolist()
cases=cols[:]
cases.remove("lexeme")
print(len(cases),", ".join(cases))

51 ai1P, ai1S, ai2P, ai2S, ai3P, ai3S, fi1P, fi1S, fi2P, fi2S, fi3P, fi3S, ii1P, ii1S, ii2P, ii2S, ii3P, ii3S, inf, is1P, is1S, is2P, is2S, is3P, is3S, pI1P, pI2P, pI2S, pP, pc1P, pc1S, pc2P, pc2S, pc3P, pc3S, pi1P, pi1S, pi2P, pi2S, pi3P, pi3S, ppFP, ppFS, ppMP, ppMS, ps1P, ps1S, ps2P, ps2S, ps3P, ps3S


In [235]:
g=nx.Graph()
for (c1,c2) in it.combinations(cases, 2):
    c1Val=paradigmes[c1].notnull()
    c2Val=paradigmes[c2].notnull()
    incompatibles=paradigmes[c1Val & c2Val & (paradigmes[c1]!=paradigmes[c2])][[c1,c2]]
    compatibles=paradigmes[c1Val & c2Val & (paradigmes[c1]==paradigmes[c2])][[c1,c2]]
    if len(incompatibles)==0 and len(compatibles)>0:
        g.add_edge(c1,c2,weight=len(compatibles))
cliques=list(nx.find_cliques(g))

In [236]:
cliques

[['ii1P', 'is1P'],
 ['pc3S', 'pc1S'],
 ['is3S', 'ai2S', 'ai3S'],
 ['pI1P', 'pi1P'],
 ['ppMS', 'ppMP'],
 ['ii3S', 'ii2S', 'ii3P'],
 ['ppFP', 'ppFS'],
 ['ii2S', 'ii1S'],
 ['fi1S', 'pc1S'],
 ['pi1S', 'pI2S'],
 ['pi2S', 'pi3S'],
 ['ps2S', 'ps3P', 'ps3S'],
 ['is2S', 'ps3S', 'ps1S'],
 ['is2S', 'is1S', 'is3P'],
 ['is2S', 'is1S', 'pi3P'],
 ['ps3P', 'ps3S', 'ps1S']]

In [237]:
cliques=sorted(cliques, key=len, reverse=True)
print(cliques)
maxLenClique=len(cliques[0])
wCliques=(cliques[:])

[['is3S', 'ai2S', 'ai3S'], ['ii3S', 'ii2S', 'ii3P'], ['ps2S', 'ps3P', 'ps3S'], ['is2S', 'ps3S', 'ps1S'], ['is2S', 'is1S', 'is3P'], ['is2S', 'is1S', 'pi3P'], ['ps3P', 'ps3S', 'ps1S'], ['ii1P', 'is1P'], ['pc3S', 'pc1S'], ['pI1P', 'pi1P'], ['ppMS', 'ppMP'], ['ppFP', 'ppFS'], ['ii2S', 'ii1S'], ['fi1S', 'pc1S'], ['pi1S', 'pI2S'], ['pi2S', 'pi3S']]


In [238]:
syncretismes=[]

def cleanCliques(lCliques):
    sCases=set(sum(lCliques,[]))
    for c in lCliques:
        wCliques.remove(c)
    for c in sCases:
        for w in wCliques:
            if c in w:
                w.remove(c)
    return
                

def addCliques(length):
    conflits=[]
    lCliques=[x for x in cliques if len(x)==length]
    if len(lCliques)>1:
        for (c1,c2) in it.combinations(lCliques, 2):
            inter=set(c1).intersection(set(c2))
            if inter:
                conflits.append([c1,c2])
    if conflits:
        ajouts=[]
        for c in lCliques:
            noConflit=True
            for conflit in conflits:
                if c in conflit:
                    noConflit=False
            if noConflit:
                ajouts.append(c)
        syncretismes.extend(ajouts)
        cleanCliques(ajouts)
        for c1,c2 in conflits:
            sC1=set(c1)
            sC2=set(c2)
            pivot=list(sC1.intersection(sC2))[0]
            dC1=sC1.difference(sC2)
            wC1=sum(g[pivot][c]["weight"] for c in dC1)
            dC2=sC2.difference(sC1)
            wC2=sum(g[pivot][c]["weight"] for c in dC2)
            print (c1,wC1,c2,wC2)
    else:
        syncretismes.extend(lCliques)
        cleanCliques(lCliques)
    return

In [239]:
for i in range(maxLenClique):
    if maxLenClique-i>1:
        print(maxLenClique-i)
        addCliques(maxLenClique-i)

print(syncretismes)
syncretiques=set()
for l in syncretismes:
    print(l)
    for c in l:
        print(c)
        syncretiques.add(c)

syncretismes,syncretiques,wCliques

3
['ps2S', 'ps3P', 'ps3S'] 493 ['is2S', 'ps3S', 'ps1S'] 253
['ps2S', 'ps3P', 'ps3S'] 184 ['ps3P', 'ps3S', 'ps1S'] 196
['is2S', 'ps3S', 'ps1S'] 9 ['is2S', 'is1S', 'is3P'] 7
['is2S', 'ps3S', 'ps1S'] 9 ['is2S', 'is1S', 'pi3P'] 10
['is2S', 'ps3S', 'ps1S'] 5 ['ps3P', 'ps3S', 'ps1S'] 255
['is2S', 'is1S', 'is3P'] 4 ['is2S', 'is1S', 'pi3P'] 7
2
['pc3S', 'pc1S'] 214 ['fi1S', 'pc1S'] 223
[['is3S', 'ai2S', 'ai3S'], ['ii3S', 'ii2S', 'ii3P'], ['ii1P', 'is1P'], ['pI1P', 'pi1P'], ['ppMS', 'ppMP'], ['ppFP', 'ppFS'], ['pi1S', 'pI2S'], ['pi2S', 'pi3S']]
['is3S', 'ai2S', 'ai3S']
is3S
ai2S
ai3S
['ii3S', 'ii2S', 'ii3P']
ii3S
ii2S
ii3P
['ii1P', 'is1P']
ii1P
is1P
['pI1P', 'pi1P']
pI1P
pi1P
['ppMS', 'ppMP']
ppMS
ppMP
['ppFP', 'ppFS']
ppFP
ppFS
['pi1S', 'pI2S']
pi1S
pI2S
['pi2S', 'pi3S']
pi2S
pi3S


([['is3S', 'ai2S', 'ai3S'],
  ['ii3S', 'ii2S', 'ii3P'],
  ['ii1P', 'is1P'],
  ['pI1P', 'pi1P'],
  ['ppMS', 'ppMP'],
  ['ppFP', 'ppFS'],
  ['pi1S', 'pI2S'],
  ['pi2S', 'pi3S']],
 {'ai2S',
  'ai3S',
  'ii1P',
  'ii2S',
  'ii3P',
  'ii3S',
  'is1P',
  'is3S',
  'pI1P',
  'pI2S',
  'pi1P',
  'pi1S',
  'pi2S',
  'pi3S',
  'ppFP',
  'ppFS',
  'ppMP',
  'ppMS'},
 [['ps2S', 'ps3P', 'ps3S'],
  ['is2S', 'ps3S', 'ps1S'],
  ['is2S', 'is1S', 'is3P'],
  ['is2S', 'is1S', 'pi3P'],
  ['ps3P', 'ps3S', 'ps1S'],
  ['pc3S', 'pc1S'],
  ['ii1S'],
  ['fi1S', 'pc1S']])

In [240]:
dExpansion={s[0]:s for s in syncretismes if len(s)>1}
dExpansion

{'is3S': ['is3S', 'ai2S', 'ai3S'],
 'ii3S': ['ii3S', 'ii2S', 'ii3P'],
 'ii1P': ['ii1P', 'is1P'],
 'pI1P': ['pI1P', 'pi1P'],
 'ppMS': ['ppMS', 'ppMP'],
 'ppFP': ['ppFP', 'ppFS'],
 'pi1S': ['pi1S', 'pI2S'],
 'pi2S': ['pi2S', 'pi3S']}

In [241]:
if "fi2S" in dExpansion:
    if "fi3S" in dExpansion["fi2S"]:
        dExpansion[u"fi3S"]=dExpansion.pop("fi2S")
if "ii1S" in dExpansion:
    if "ii3S" in dExpansion["ii1S"]:
        dExpansion[u"ii3S"]=dExpansion.pop("ii1S")
if "pc2S" in dExpansion:
    if "pc3S" in dExpansion["pc2S"]:
        dExpansion[u"pc3S"]=dExpansion.pop("pc2S")
if "ps2S" in dExpansion:
    if "ps3S" in dExpansion["ps2S"]:
        dExpansion[u"ps3S"]=dExpansion.pop("ps2S")        
if "pi2S" in dExpansion:
    if "pi3S" in dExpansion["pi2S"]:
        dExpansion[u"pi3S"]=dExpansion.pop("pi2S")
if "ppMP" in dExpansion:
    if "ppMS" in dExpansion["ppMP"]:
        dExpansion[u"ppMS"]=dExpansion.pop("ppMP")

In [242]:
dExpansion

{'is3S': ['is3S', 'ai2S', 'ai3S'],
 'ii3S': ['ii3S', 'ii2S', 'ii3P'],
 'ii1P': ['ii1P', 'is1P'],
 'pI1P': ['pI1P', 'pi1P'],
 'ppMS': ['ppMS', 'ppMP'],
 'ppFP': ['ppFP', 'ppFS'],
 'pi1S': ['pi1S', 'pI2S'],
 'pi3S': ['pi2S', 'pi3S']}

In [243]:
print([c for c in cols if c not in syncretiques])
omps=paradigmes[[c for c in cols if c not in syncretiques]].copy()

['lexeme', 'ai1P', 'ai1S', 'ai2P', 'ai3P', 'fi1P', 'fi1S', 'fi2P', 'fi2S', 'fi3P', 'fi3S', 'ii1S', 'ii2P', 'inf', 'is1S', 'is2P', 'is2S', 'is3P', 'pI2P', 'pP', 'pc1P', 'pc1S', 'pc2P', 'pc2S', 'pc3P', 'pc3S', 'pi2P', 'pi3P', 'ps1P', 'ps1S', 'ps2P', 'ps2S', 'ps3P', 'ps3S']


In [244]:
def fusionFormes(row):
    result=np.nan
    for c in row:
        if c==c:
            result=c
            break
    return result

In [245]:
# paradigmes[["ppMS","ppMP"]].apply(fusionFormes,axis=1)

In [246]:
for s in dExpansion:
    print (s,dExpansion[s])
    omps[s]=paradigmes[dExpansion[s]].apply(fusionFormes,axis=1)

is3S ['is3S', 'ai2S', 'ai3S']
ii3S ['ii3S', 'ii2S', 'ii3P']
ii1P ['ii1P', 'is1P']
pI1P ['pI1P', 'pi1P']
ppMS ['ppMS', 'ppMP']
ppFP ['ppFP', 'ppFS']
pi1S ['pi1S', 'pI2S']
pi3S ['pi2S', 'pi3S']


In [247]:
with open(rep+fParadigmes.replace(".csv","-omp.yaml"),"w") as outFile:
    yaml.safe_dump(dExpansion,outFile)

In [248]:
omps.dropna(thresh=2).to_csv(rep+fParadigmes.replace(".csv","-omp.csv"),encoding="utf8",sep=";",index=None)

In [249]:
if num<9:
    print(num)
    num+=1
    ding()
else:
    num=0
    ding()
    ding()
    ding()
numOMP=num
%store numOMP

2
Stored 'numOMP' (int)
